# 04. GitHub Statistical Tests (RQ1)

3-way classification(Anime/Default/Photo) profile type between GitHub activity metric differences are statistically significant test.

| test | purpose | notes |
|------|------|------|
| Kruskal-Wallis H | 3group between median difference | non-parametric test |
| Dunn's test | post-hoc test (which pairs differ) | Bonferroni correction |
| Mann-Whitney U | Anime vs non-anime binary comparison | |
| χ² independence test | PFP type × activity_grade | categorical variable |
| effect size | Eta-squared, Cliff's delta | practical difference magnitude |

**Input**: `data/processed/classified_3cat.csv`

In [ ]:
import json
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

matplotlib.rcParams['font.family'] = 'AppleGothic'
matplotlib.rcParams['axes.unicode_minus'] = False

PROJECT_DIR = Path('../')
DATA_DIR = PROJECT_DIR / 'data'
PROC_DIR = DATA_DIR / 'processed'
RAW_DIR = DATA_DIR / 'raw'
df = pd.read_csv(PROC_DIR / 'classified_3cat.csv')

with open(RAW_DIR / 'enriched_users.json') as f:
    enriched = json.load(f)

metrics_df = pd.DataFrame([
    {
        'uid': u['user_id'],
        'followers': u.get('followers', 0),
        'public_repos': u.get('public_repos', 0),
        'total_stars': u.get('repos', {}).get('total_stars_received', 0),
        'total_forks': u.get('repos', {}).get('total_forks_received', 0),
        'activity_grade': u.get('activity_grade'),
        'sampling_group': u.get('sampling_group'),
    }
    for u in enriched.values()
])
df = df.merge(metrics_df, on='uid', how='left')

contrib_path = RAW_DIR / 'contributions.json'
if contrib_path.exists():
    with open(contrib_path) as f:
        contributions = json.load(f)
    contrib_df = pd.DataFrame([
        {
            'uid': int(uid),
            'commits': c.get('commits', 0),
            'prs': c.get('prs', 0),
            'issues': c.get('issues', 0),
            'reviews': c.get('reviews', 0),
            'total_contributions': c.get('total', 0),
        }
        for uid, c in contributions.items()
    ])
    df = df.merge(contrib_df, on='uid', how='left')

anime = df[df['profile_type'] == 'Anime']
default = df[df['profile_type'] == 'Default']
normal = df[df['profile_type'] == 'Photo']

print(f'Anime: {len(anime)}, Default: {len(default)}, Photo: {len(normal)}')

## 1. Kruskal-Wallis H Test (3-group comparison)

In [ ]:
metrics = ['followers', 'total_stars', 'public_repos', 'total_forks']
if 'commits' in df.columns:
    metrics += ['commits', 'total_contributions']

kw_results = []

for m in metrics:
    h_stat, p_val = stats.kruskal(
        anime[m].dropna(),
        default[m].dropna(),
        normal[m].dropna()
    )
    # Eta-squared effect size: η² = (H - k + 1) / (n - k)
    n = len(df[m].dropna())
    k = 3
    eta_sq = (h_stat - k + 1) / (n - k)
    eta_sq = max(0, eta_sq)  # clamp to 0
    
    sig = '***' if p_val < 0.001 else '**' if p_val < 0.01 else '*' if p_val < 0.05 else 'ns'
    kw_results.append({
        'metric': m,
        'H-statistic': round(h_stat, 2),
        'p-value': f'{p_val:.2e}',
        'η²': round(eta_sq, 4),
        'significance': sig
    })

kw_df = pd.DataFrame(kw_results)
print('=== Kruskal-Wallis H test (3group: Anime/Default/Photo) ===')
kw_df

## 2. Dunn's Post-hoc Test

In [ ]:
from itertools import combinations

groups = {'Anime': anime, 'Default': default, 'Photo': normal}
pairs = list(combinations(groups.keys(), 2))

dunn_results = []
for m in metrics:
    for g1, g2 in pairs:
        u_stat, p_val = stats.mannwhitneyu(
            groups[g1][m].dropna(),
            groups[g2][m].dropna(),
            alternative='two-sided'
        )
        # Bonferroni correction (3 pairs)
        p_corrected = min(p_val * 3, 1.0)
        
        # Cliff's delta effect size
        n1, n2 = len(groups[g1][m].dropna()), len(groups[g2][m].dropna())
        cliff_d = (2 * u_stat) / (n1 * n2) - 1
        
        # Effect size interpretation
        abs_d = abs(cliff_d)
        effect = 'large' if abs_d > 0.474 else 'medium' if abs_d > 0.33 else 'small' if abs_d > 0.147 else 'negligible'
        
        sig = '***' if p_corrected < 0.001 else '**' if p_corrected < 0.01 else '*' if p_corrected < 0.05 else 'ns'
        dunn_results.append({
            'metric': m,
            'comparison': f'{g1} vs {g2}',
            'U-statistic': int(u_stat),
            'p (Bonferroni)': f'{p_corrected:.2e}',
            "Cliff's δ": round(cliff_d, 3),
            'effect size': effect,
            'significance': sig
        })

dunn_df = pd.DataFrame(dunn_results)
print('=== post-hoc test (Mann-Whitney U + Bonferroni) ===')
dunn_df

## 3. Anime vs Non-Anime Binary Comparison

In [ ]:
non_anime = df[df['profile_type'] != 'Anime']

binary_results = []
for m in metrics:
    u_stat, p_val = stats.mannwhitneyu(
        anime[m].dropna(),
        non_anime[m].dropna(),
        alternative='two-sided'
    )
    n1, n2 = len(anime[m].dropna()), len(non_anime[m].dropna())
    cliff_d = (2 * u_stat) / (n1 * n2) - 1
    abs_d = abs(cliff_d)
    effect = 'large' if abs_d > 0.474 else 'medium' if abs_d > 0.33 else 'small' if abs_d > 0.147 else 'negligible'
    sig = '***' if p_val < 0.001 else '**' if p_val < 0.01 else '*' if p_val < 0.05 else 'ns'
    
    binary_results.append({
        'metric': m,
        'Anime median': anime[m].median(),
        'non-anime median': non_anime[m].median(),
        'U-statistic': int(u_stat),
        'p-value': f'{p_val:.2e}',
        "Cliff's δ": round(cliff_d, 3),
        'effect size': effect,
        'significance': sig
    })

binary_df = pd.DataFrame(binary_results)
print('=== Anime vs non-anime binary comparison ===')
binary_df

## 4. χ² independence test (PFP type × Activity Grade)

In [ ]:
contingency = pd.crosstab(df['profile_type'], df['activity_grade'])
print('contingency table:')
print(contingency)

chi2, p_val, dof, expected = stats.chi2_contingency(contingency)
# Cramér's V
n = contingency.sum().sum()
k = min(contingency.shape) - 1
cramers_v = np.sqrt(chi2 / (n * k))

print(f'\n=== χ² independence test ===')
print(f'χ² = {chi2:.2f}')
print(f'p-value = {p_val:.2e}')
print(f'degrees of freedom = {dof}')
print(f"Cramér's V = {cramers_v:.4f}")
print(f'significance: {"***" if p_val < 0.001 else "**" if p_val < 0.01 else "*" if p_val < 0.05 else "ns"}')

## 5. result summary visualization

In [ ]:
# effect size heatmap
pivot = dunn_df.pivot(index='comparison', columns='metric', values="Cliff's δ")

fig, ax = plt.subplots(figsize=(8, 4))
sns.heatmap(pivot.astype(float), annot=True, fmt='.3f', cmap='RdBu_r',
            center=0, vmin=-0.5, vmax=0.5, ax=ax)
ax.set_title("Cliff's δ effect size (post-hoc test)", fontsize=14)
plt.tight_layout()
plt.savefig(FIGURE_DIR / 'effect_size_heatmap.png', dpi=150)
plt.show()

In [ ]:
# save all results
with pd.ExcelWriter(PROC_DIR / 'statistical_results.xlsx') as writer:
    kw_df.to_excel(writer, sheet_name='Kruskal-Wallis', index=False)
    dunn_df.to_excel(writer, sheet_name='Posthoc_Dunn', index=False)
    binary_df.to_excel(writer, sheet_name='Anime_vs_NonAnime', index=False)

print('Statistical results saved: statistical_results.xlsx')

## 6. Conclusion

Interpretation of RQ1 based on results above:

- Were significant differences found? → p-value check
- Are the differences practically meaningful? → effect size(Cliff's δ, Cramér's V) check
- **Note**: correlation and not causation. confounding variables(culture, experience, etc.) may have influence